# UnifyWeaver 中的高级递归模式

本笔记本演示了 UnifyWeaver 能够检测和优化的四种主要递归模式：

1. **尾递归 (Tail Recursion)** - 带累加器的迭代循环
2. **线性递归 (Linear Recursion)** - 带记忆化的单次递归调用
3. **树形递归 (Tree Recursion)** - 在结构各部分上的多次递归调用
4. **互递归 (Mutual Recursion)** - 循环相互调用的谓词

## 学习目标

- 理解不同的递归模式
- 了解 UnifyWeaver 如何检测和优化每种模式
- 比较性能特征
- 掌握何时使用每种模式

## 环境配置

初始化 UnifyWeaver 环境。

In [ ]:
% Load initialization
['../init'].

% Load necessary modules
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## 模式 1：尾递归 (Tail Recursion)

尾递归使用累加器传递中间结果，且递归调用是函数中的**最后一个操作**。

### 示例：统计列表中的元素个数

In [ ]:
% Define tail-recursive count_items
:- dynamic count_items/3.

% Base case: empty list, return accumulator
count_items([], Acc, Acc).

% Recursive case: increment accumulator, recurse on tail
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← Tail position!

### 在 Prolog 中测试

In [ ]:
% Test: count items in [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### 检查模式检测结果

In [ ]:
% Check if detected as tail recursive
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### 编译为 Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### 测试生成的 Bash 代码

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Counting items in [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## 模式 2：线性递归 (Linear Recursion)

线性递归在每个子句中包含**恰好一个**递归调用，计算在递归调用返回后进行。

### 示例：阶乘 (Factorial)

In [ ]:
% Define factorial
:- dynamic factorial/2.

% Base case
factorial(0, 1).

% Recursive case: exactly ONE recursive call
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← One recursive call
    F is N * F1.        % ← Computation after call

### 在 Prolog 中测试

In [ ]:
% Test: factorial of 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### 检查模式检测结果

In [ ]:
% Check if detected as linear recursive
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### 编译为 Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Keep function definitions only; Brush treats sourced scripts as direct execution
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### 测试生成的 Bash 代码

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Factorial of 5:"
factorial 5 ""
echo ""
echo "Factorial of 10:"
factorial 10 ""

## 模式 3：树形递归 (Tree Recursion)

树形递归进行**多次**递归调用以处理结构的不同部分。

### 示例：树节点求和 (Tree Sum)

In [ ]:
% Define tree_sum for binary trees
% Tree format: [Value, LeftSubtree, RightSubtree] or []
:- dynamic tree_sum/2.

% Base case: empty tree has sum 0
tree_sum([], 0).

% Recursive case: sum = value + left_sum + right_sum
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← First recursive call
    tree_sum(R, RS),   % ← Second recursive call
    Sum is V + LS + RS.

### 在 Prolog 中测试

In [ ]:
% Test: tree_sum of [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### 编译为 Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### 测试生成的 Bash 代码

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Tree sum of [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## 模式 4：互递归 (Mutual Recursion)

当两个或多个谓词在一个循环中相互调用时，就会发生互递归。

### 示例：偶数 (Even) 与奇数 (Odd)

In [ ]:
% Define mutually recursive is_even and is_odd
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even base case
is_even(0).

% is_even recursive: N is even if N-1 is odd
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Calls is_odd

% is_odd base case
is_odd(1).

% is_odd recursive: N is odd if N-1 is even
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Calls is_even

### 在 Prolog 中测试

In [ ]:
% Test even/odd
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### 检查互递归

In [ ]:
% Build call graph and find SCCs
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### 编译为 Bash

In [ ]:
% Compile the mutual recursion group
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### 测试生成的 Bash 代码

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "Testing is_even and is_odd:"
is_even 0 >/dev/null && echo "✓ 0 is even"
is_even 4 >/dev/null && echo "✓ 4 is even"
is_odd 3 >/dev/null && echo "✓ 3 is odd"
is_odd 7 >/dev/null && echo "✓ 7 is odd"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 is not even"

## 模式比较

比较每种模式的特性：

| 模式 | 递归调用 | 优化方式 | 空间复杂度 | 最适合场景 |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **尾递归** | 1 次（处于尾部位置） | 迭代循环 | O(1) | 累加器、线性扫描 |
| **线性递归** | 1 次（任意位置） | 折叠 (Fold) + 记忆化 | O(n) 记忆表 | 斐波那契、阶乘 |
| **树形递归** | 2 次以上（结构各部分） | 结构解构 | O(深度) 栈 | 树/图操作 |
| **互递归** | 1 次以上（跨谓词） | 共享记忆化 | O(n) 共享表 | 奇/偶判断、相互定义 |

## 模式检测顺序

UnifyWeaver 按照以下顺序尝试匹配模式：

1. **尾递归**（最高效）
2. **线性递归**（除非被禁用）
3. **树形递归**（结构性）
4. **互递归**（强连通分量 SCC 检测）
5. **基础递归**（默认回退方案）

你可以使用 `forbid_linear_recursion/1` 影响检测流程。

## 练习：动手试试！

尝试定义并编译以下谓词：

### 1. 尾递归求和
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. 线性递归斐波那契
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. 树的高度
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% Your code here!


## 总结

在本笔记本中，你学习了：

✅ UnifyWeaver 中的四种主要递归模式

✅ 如何在 Prolog 中定义每种模式

✅ UnifyWeaver 如何检测和优化每种模式

✅ 每种模式的性能特征

✅ 何时使用对应模式

## 后续步骤

继续前往 **笔记本 3：调用图可视化**，学习高级代码分析与可视化！